# FMA Small Dataset Downloader

Downloads the [FMA Small](https://github.com/mdeff/fma) dataset (8,000 tracks of 30s, 8 genres, ~7.2 GiB) and its metadata.

- Songs are extracted to `data/fma_small/`
- Metadata is extracted to `data/fma_metadata/`
- Zips are cached in `temp/` — reused on re-runs

In [1]:
import hashlib
import zipfile
import urllib.request
from pathlib import Path

In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT       = Path("data")
SONGS_DIR  = ROOT / "fma_small"
META_DIR   = ROOT / "fma_metadata"
TEMP_DIR   = Path("temp")

AUDIO_ZIP  = TEMP_DIR / "fma_small.zip"
META_ZIP   = TEMP_DIR / "fma_metadata.zip"

AUDIO_URL  = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
META_URL   = "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip"

# SHA-1 checksums from the official FMA README
AUDIO_SHA1 = "ade154f733639d52e35e32f5593efe5be76c6d70"
META_SHA1  = "f0df49ffe5f2a6008d7dc83c6915b31835dfe733"

# Expected number of MP3 tracks in fma_small
EXPECTED_TRACK_COUNT = 8000

In [3]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def sha1_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while buf := f.read(chunk):
            h.update(buf)
    return h.hexdigest()


def download(url: str, dest: Path) -> None:
    print(f"Downloading {url}")
    print(f"  → {dest}")

    def _progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = min(downloaded / total_size * 100, 100)
            gib = total_size / (1 << 30)
            print(f"\r  {pct:5.1f}%  ({gib:.2f} GiB total)", end="", flush=True)

    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print()  # newline after progress


def verify(path: Path, expected_sha1: str) -> bool:
    print(f"Verifying checksum for {path.name} …", end=" ", flush=True)
    digest = sha1_file(path)
    ok = digest == expected_sha1
    print("OK" if ok else f"MISMATCH (got {digest})")
    return ok


def count_mp3s(directory: Path) -> int:
    return sum(1 for _ in directory.rglob("*.mp3"))


def ensure_zip(zip_path: Path, url: str, sha1: str) -> None:
    """Download the zip only if not already cached, then verify."""
    if zip_path.exists():
        print(f"Found cached zip: {zip_path}")
        if not verify(zip_path, sha1):
            print("Checksum failed — re-downloading …")
            zip_path.unlink()
            download(url, zip_path)
            assert verify(zip_path, sha1), "Checksum still failed after re-download!"
    else:
        download(url, zip_path)
        assert verify(zip_path, sha1), "Checksum failed after download!"


def extract_zip(zip_path: Path, dest: Path) -> None:
    print(f"Extracting {zip_path.name} → {dest} …")
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest)
    print("Extraction complete.")

In [4]:
# ── Create directories ────────────────────────────────────────────────────────
ROOT.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)
print(f"Directories ready: {ROOT}/  {TEMP_DIR}/")

Directories ready: data/  temp/


In [5]:
# ── Audio (fma_small) ─────────────────────────────────────────────────────────

mp3_count = count_mp3s(SONGS_DIR) if SONGS_DIR.exists() else 0
print(f"Songs already on disk: {mp3_count} / {EXPECTED_TRACK_COUNT}")

if mp3_count >= EXPECTED_TRACK_COUNT:
    print("All tracks present — skipping audio download & extraction.")
else:
    ensure_zip(AUDIO_ZIP, AUDIO_URL, AUDIO_SHA1)
    extract_zip(AUDIO_ZIP, ROOT)
    mp3_count = count_mp3s(SONGS_DIR)
    print(f"Tracks on disk after extraction: {mp3_count}")
# clean up zip files 
for f in [AUDIO_ZIP, META_ZIP]:
    if f.exists():
        f.unlink()


Songs already on disk: 0 / 8000
  → temp/fma_small.zip


  100.0%  (7.15 GiB total)
Verifying checksum for fma_small.zip … OK
Extracting fma_small.zip → data …
Extraction complete.
Tracks on disk after extraction: 8000


In [6]:
# ── Metadata (fma_metadata) ───────────────────────────────────────────────────

# Key files produced by the metadata archive
META_FILES = ["tracks.csv", "genres.csv", "features.csv", "echonest.csv"]
meta_present = all((META_DIR / f).exists() for f in META_FILES)

if meta_present:
    print("Metadata already present — skipping metadata download & extraction.")
else:
    ensure_zip(META_ZIP, META_URL, META_SHA1)
    extract_zip(META_ZIP, ROOT)
    print("Metadata files:", [f for f in META_FILES if (META_DIR / f).exists()])

  → temp/fma_metadata.zip
  100.0%  (0.33 GiB total)
Verifying checksum for fma_metadata.zip … OK
Extracting fma_metadata.zip → data …
Extraction complete.
Metadata files: ['tracks.csv', 'genres.csv', 'features.csv', 'echonest.csv']


In [7]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 50)
print(f"Songs dir  : {SONGS_DIR.resolve()}")
print(f"MP3 tracks : {count_mp3s(SONGS_DIR) if SONGS_DIR.exists() else 0}")
print(f"Metadata   : {META_DIR.resolve()}")
for f in META_FILES:
    p = META_DIR / f
    status = f"{p.stat().st_size / (1 << 20):.1f} MiB" if p.exists() else "MISSING"
    print(f"  {f:<20} {status}")
print("=" * 50)

Songs dir  : /home/nathan/Documents/PlaylistGenerator/data/fma_small
MP3 tracks : 8000
Metadata   : /home/nathan/Documents/PlaylistGenerator/data/fma_metadata
  tracks.csv           248.4 MiB
  genres.csv           0.0 MiB
  features.csv         907.1 MiB
  echonest.csv         42.0 MiB


In [8]:
import pandas as pd
print("=" * 50)

# for each file, its csv columns 
for f in META_FILES:
    p = META_DIR / f
    if p.exists():
        df = pd.read_csv(p)
        print(f"Columns in {f}: {df.columns.tolist()}")
    else:
        print(f"{f} is missing")

/tmp/ipykernel_6772/3121055587.py:8: DtypeWarning: Columns (0,1,5,6,8,12,18,20,21,22,24,33,34,38,39,44,47,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(p)


Columns in tracks.csv: ['Unnamed: 0', 'album', 'album.1', 'album.2', 'album.3', 'album.4', 'album.5', 'album.6', 'album.7', 'album.8', 'album.9', 'album.10', 'album.11', 'album.12', 'artist', 'artist.1', 'artist.2', 'artist.3', 'artist.4', 'artist.5', 'artist.6', 'artist.7', 'artist.8', 'artist.9', 'artist.10', 'artist.11', 'artist.12', 'artist.13', 'artist.14', 'artist.15', 'artist.16', 'set', 'set.1', 'track', 'track.1', 'track.2', 'track.3', 'track.4', 'track.5', 'track.6', 'track.7', 'track.8', 'track.9', 'track.10', 'track.11', 'track.12', 'track.13', 'track.14', 'track.15', 'track.16', 'track.17', 'track.18', 'track.19']
Columns in genres.csv: ['genre_id', '#tracks', 'parent', 'title', 'top_level']


/tmp/ipykernel_6772/3121055587.py:8: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,26

Columns in features.csv: ['feature', 'chroma_cens', 'chroma_cens.1', 'chroma_cens.2', 'chroma_cens.3', 'chroma_cens.4', 'chroma_cens.5', 'chroma_cens.6', 'chroma_cens.7', 'chroma_cens.8', 'chroma_cens.9', 'chroma_cens.10', 'chroma_cens.11', 'chroma_cens.12', 'chroma_cens.13', 'chroma_cens.14', 'chroma_cens.15', 'chroma_cens.16', 'chroma_cens.17', 'chroma_cens.18', 'chroma_cens.19', 'chroma_cens.20', 'chroma_cens.21', 'chroma_cens.22', 'chroma_cens.23', 'chroma_cens.24', 'chroma_cens.25', 'chroma_cens.26', 'chroma_cens.27', 'chroma_cens.28', 'chroma_cens.29', 'chroma_cens.30', 'chroma_cens.31', 'chroma_cens.32', 'chroma_cens.33', 'chroma_cens.34', 'chroma_cens.35', 'chroma_cens.36', 'chroma_cens.37', 'chroma_cens.38', 'chroma_cens.39', 'chroma_cens.40', 'chroma_cens.41', 'chroma_cens.42', 'chroma_cens.43', 'chroma_cens.44', 'chroma_cens.45', 'chroma_cens.46', 'chroma_cens.47', 'chroma_cens.48', 'chroma_cens.49', 'chroma_cens.50', 'chroma_cens.51', 'chroma_cens.52', 'chroma_cens.53', 'ch

/tmp/ipykernel_6772/3121055587.py:8: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,11,13,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249) have mixed types. Specify dtype option on import or set low_mem

Columns in echonest.csv: ['Unnamed: 0', 'echonest', 'echonest.1', 'echonest.2', 'echonest.3', 'echonest.4', 'echonest.5', 'echonest.6', 'echonest.7', 'echonest.8', 'echonest.9', 'echonest.10', 'echonest.11', 'echonest.12', 'echonest.13', 'echonest.14', 'echonest.15', 'echonest.16', 'echonest.17', 'echonest.18', 'echonest.19', 'echonest.20', 'echonest.21', 'echonest.22', 'echonest.23', 'echonest.24', 'echonest.25', 'echonest.26', 'echonest.27', 'echonest.28', 'echonest.29', 'echonest.30', 'echonest.31', 'echonest.32', 'echonest.33', 'echonest.34', 'echonest.35', 'echonest.36', 'echonest.37', 'echonest.38', 'echonest.39', 'echonest.40', 'echonest.41', 'echonest.42', 'echonest.43', 'echonest.44', 'echonest.45', 'echonest.46', 'echonest.47', 'echonest.48', 'echonest.49', 'echonest.50', 'echonest.51', 'echonest.52', 'echonest.53', 'echonest.54', 'echonest.55', 'echonest.56', 'echonest.57', 'echonest.58', 'echonest.59', 'echonest.60', 'echonest.61', 'echonest.62', 'echonest.63', 'echonest.64